# 1장 — 큰 그림: 키우면 좋아진다, 그런데 얼마나? 🗺️

"모델을 키운다"에 섞여 있는 **세 재료**를 가르고, 훈련 예산 계산서를 직접 뽑아 봅니다.

- **파라미터 N**: 가중치 개수(몸집). 9부 1장에서 공식으로 세어 본 그 값
- **데이터 D**: 훈련 토큰 수(먹인 양). 4부 2장의 그 토큰
- **컴퓨트 C**: 총 연산량(훈련 예산). 어림식 **C ≈ 6·N·D**

웹 챕터의 다이얼 랩과 같은 공식·같은 상수라 숫자가 정확히 일치합니다.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch1')

## 1. 훈련 예산 계산서: C ≈ 6·N·D

토큰 하나가 파라미터 하나를 지날 때 **순전파 약 2회 + 역전파 약 4회 ≈ 6회** 연산이 든다는
어림입니다. GPU-일 환산 가정: A100 bf16 최대 312 TFLOPS × 실효 가동률 50%.

In [ ]:
EFF_FLOPS = 312e12 * 0.5      # A100 실효 성능 (FLOPs/초)
DAY = 86400

def budget(n, d):
    """훈련 예산 — (C, GPU-일). C ≈ 6·N·D."""
    c = 6 * n * d
    return c, c / EFF_FLOPS / DAY

presets = [
    ('GPT-2급', 1.5e9, 10e9),
    ('7B급   ', 7e9, 1e12),
    ('70B급  ', 70e9, 2e12),
]

print('모델    |      N |     D  |        C (FLOPs) | A100 1장 | 1,000장')
print('-' * 72)
for name, n, d in presets:
    c, days = budget(n, d)
    one = f'{days/365.25:.1f}년' if days >= 365.25 else f'{days:.1f}일'
    k = f'{days/1000:.1f}일' if days / 1000 >= 1 else f'{days/1000*24:.1f}시간'
    print(f'{name} | {n/1e9:>5.1f}B | {d/1e9:>5.0f}B | {c:.1e} | {one:>8} | {k:>7}')

# 웹 랩·본문 수치와 못박기
c2, d2 = budget(1.5e9, 10e9)
c7, d7 = budget(7e9, 1e12)
c70, d70 = budget(70e9, 2e12)
assert c2 == 9e19 and round(d2, 1) == 6.7
assert c7 == 4.2e22 and round(d7) == 3116 and abs(d7 / 365.25 - 8.5) < 0.05, '7B급 = 1장 8.5년'
assert c70 == 8.4e23 and round(d70) == 62322 and abs(d70 / 365.25 - 170.6) < 0.1
print()
print('✔ 웹 랩과 동일: 7B·1T = 4.2×10²² FLOPs, A100 1장 8.5년(1,000장 3.1일)')

## 2. 곱의 성질: 재료를 2배씩 올리면 예산은 4배

C는 N과 D의 **곱**이라, 두 재료를 함께 키우면 예산이 제곱으로 뜁니다.
지수 계산 감각(2부 7장)도 함께 확인해 두죠: 앞자리끼리 곱하고, 지수끼리 더한다.

In [ ]:
n, d = 7e9, 1e12
c0, _ = budget(n, d)
c1, _ = budget(2 * n, 2 * d)
assert c1 == 4 * c0, 'N·D 각 2배 → C는 4배 (곱!)'
print(f'C(7B, 1T)   = {c0:.1e}')
print(f'C(14B, 2T)  = {c1:.1e}  ← 4배')

# 지수로 손 계산: 6×7 = 42, 10^9 × 10^12 = 10^21 → 42×10^21 = 4.2×10^22
assert 6 * 7 == 42 and 9 + 12 == 21
assert abs(42e21 - 4.2e22) < 1e7
print()
print('✔ 손 계산 검증: 6×7=42, 지수 9+12=21 → 4.2×10²² (StepQuiz 와 동일)')

## 3. 그림으로: 예산의 규모 감각

왼쪽: 프리셋별 훈련 예산 C(로그 축, 눈금이 10배씩!). 오른쪽: A100 1장으로 걸리는 시간.

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

names = [p[0].strip() for p in presets]
cs = [budget(n, d)[0] for _, n, d in presets]
ds = [budget(n, d)[1] for _, n, d in presets]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.bar(names, cs, color='#6366f1')
ax1.set_yscale('log')
ax1.set_title('훈련 예산 C = 6·N·D (FLOPs, 로그 축)')
for i, c in enumerate(cs):
    ax1.text(i, c * 1.3, f'{c:.0e}', ha='center', fontsize=9)

ax2.bar(names, [d / 365.25 for d in ds], color='#f59e0b')
ax2.set_yscale('log')
ax2.set_title('A100 GPU 1장으로 걸리는 시간 (년, 로그 축)')
ax2.axhline(1, color='#94a3b8', ls='--', lw=1)
for i, d in enumerate(ds):
    y = d / 365.25
    ax2.text(i, y * 1.3, f'{y:.2f}년' if y >= 1 else f'{d:.1f}일', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print('로그 축이 아니면 GPT-2급 막대는 보이지도 않습니다. 곱의 세계를 그리는 법이 2장 주제.')

## 정리: 오늘 가져갈 세 문장

1. "키운다"는 세 재료로 갈라진다: **몸집 N · 먹인 양 D · 예산 C**.
2. **C ≈ 6·N·D**: 곱이라서 함께 키우면 예산이 제곱으로 뛴다 (7B·1T = A100 1장 8.5년).
3. 같은 예산으로 "크게 잠깐 vs 작게 오래". 이 배분 문제의 답(황금비)이 **4장 Chinchilla**.

다음 **2장(곱의 세계)**: 10배씩 뛰는 수들을 다루는 도구, 로그 눈금과 멱법칙 y = a·xᵇ.